# 00 - Tensor Fundamentals
Goal: understand tensors, shapes, dtypes, and matrix ops - the building blocks of everything in PyTorch.

In [15]:
import torch

# Creating tensors
a = torch.tensor([[1.0, 2.0], [3.0, 4.0]])   # from data
b = torch.zeros(2, 3)                          # shape (2, 3)
c = torch.randn(2, 3)                          # random normal - how weights get initialised
d = torch.arange(10)                           # 0..9

print(a.shape, a.dtype)

torch.Size([2, 2]) torch.float32


Default dtype for Pytorch is `float32` used for model weights and inputs. Class **labels** must be `int64 (long)`, because CrossEntropyLoss requires it.

In [11]:
x = torch.randn(4, 1, 28, 28)   # pretend: batch of 4 grayscale 28x28 images

print(x.shape)                   # torch.Size([4, 1, 28, 28])
flat = x.reshape(4, -1)          # -1 means "compute the dimension from total count"
print(flat.shape)                # torch.Size([4, 784])

torch.Size([4, 1, 28, 28])
torch.Size([4, 784])


If an image is `1 × 28 × 28`, it has `1 × 28 × 28 = 784` pixel values.
Flattening unrolls the 2-D grid into one long vector of 784 numbers, because a fully connected layer expects each sample as a flat vector, not a grid.

**Rule:** reshape must preserve the total element count.
A batch of shape `(4, 1, 28, 28)` has `4 × 1 × 28 × 28 = 3136` elements - every reshape of it starts from that same total.

**`-1` rule:** `-1` = total elements ÷ product of the specified dims. Only one `-1` per reshape.
e.g. `reshape(2, -1)` → `3136 ÷ 2 = 1568`.

**Idiom:** use `x.reshape(x.size(0), -1)` - batch sizes vary (the last batch of an epoch is usually smaller), so hardcoding breaks.

`reshape(-1)` is  a 1-D bare vector whereas `reshape(1,-1)` is 2-D; a batch containing one row.

In [12]:
u = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
v = torch.tensor([[10.0, 20.0], [30.0, 40.0]])

print(u * v)    # elementwise: [[10, 40], [90, 160]]
print(u @ v)    # matrix multiply: [[70, 100], [150, 220]]

tensor([[ 10.,  40.],
        [ 90., 160.]])
tensor([[ 70., 100.],
        [150., 220.]])


In [13]:
batch = torch.randn(4, 784)     # 4 samples
bias  = torch.randn(784)        # one bias vector
print((batch + bias).shape)     # (4, 784) — bias added to every row

torch.Size([4, 784])


#### Linear layer - basic building block of neural networks

In [14]:
x = torch.randn(4, 784)          # batch of 4 flattened images
W = torch.randn(128, 784)        # weights: 784 inputs -> 128 outputs
b = torch.randn(128)             # bias

y = x @ W.T + b
print(y.shape)                   # (4, 128)

torch.Size([4, 128])


##### Manual linear layer - what `nn.Linear` really does

`y = x @ W.T + b`

- **784 is forced** (input size = pixels); **128 is chosen** - it's the number of neurons in the layer, a design decision.
- Each neuron has its own 784 weights → `W` is `(128, 784)`: one **row per neuron**, i.e. `(out_features, in_features)` - same order as `nn.Linear(784, 128)` stores it.
- Inner dims must match and are consumed.
  `x` is `(4, 784)`, so we need `(784, 128)` on the right → hence the transpose `W.T`.
  Result: `(4, 784) @ (784, 128) → (4, 128)` = (batch, neurons).
- `b` is `(128,)` — **one bias per neuron**. Broadcasting adds it to every row of the batch.
- `nn.Linear(784, 128)` just stores `W` and `b` and runs this exact line. No magic.

The layer represents each image as 128 learned features instead of 784 raw pixels.

## Exercises

### Ex 1: per-image mean
Goal: 8 numbers, one per image. Rule: dims I list in `.mean(dim=...)` get destroyed; dims I omit survive.

In [36]:
imgs = torch.randn(8, 1, 28, 28)
print(imgs.shape)                        # (8, 1, 28, 28)


per_image_mean = imgs.mean(dim=(1, 2, 3))

print(per_image_mean.shape)              # (8,) - one mean per image
print(per_image_mean)

torch.Size([8, 1, 28, 28])
torch.Size([8])
tensor([ 0.0251, -0.0408, -0.0459, -0.0505, -0.0138, -0.0398, -0.0706,  0.0595])


### Ex 2: manual linear layer, 10 classes
W is (10, 784) - one row per neuron. Predicted output shape: (8, 10).

In [37]:
x = torch.randn(8, 784)      # 8 flattened images
W = torch.randn(10, 784)     # 10 classes → 10 neurons, 784 inputs each
b = torch.randn(10)          # one bias per neuron

y = x @ W.T + b
print(y.shape)               # (8, 10) — 10 class scores per image

torch.Size([8, 10])


### Ex 3: dtype of int ÷ int
Prediction: float32. Why good: prevents silent truncation (1/3 would become 0 as ints).

In [38]:
result = torch.tensor([1, 2]) / torch.tensor([3, 4])
# my prediction of result.dtype: float32 -> correct
print(result, result.dtype)

tensor([0.3333, 0.5000]) torch.float32
